# 1. Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import keras
from keras import activations as activations
from keras import applications as applications
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import random
import time
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


# 2. Get Data

## Climate Data Time-Series

Dataset used for this mini-project is Jena Climate dataset recorded by the
[Max Planck Institute for Biogeochemistry](https://www.bgc-jena.mpg.de/wetter/).
The dataset consists of 14 features recorded once per 10 minutes.

**Location**: Weather Station, Max Planck Institute for Biogeochemistry
in Jena, Germany

**Time-frame Considered**: Jan 10, 2009 - December 31, 2016


The table below shows the column names, their value formats, and their description.

Index| Features      |Format             |Description
-----|---------------|-------------------|-----------------------
1    |Date Time      |01.01.2009 00:10:00|Date-time reference
2    |p (mbar)       |996.52             |The pascal SI derived unit of pressure used to quantify internal pressure. Meteorological reports typically state atmospheric pressure in millibars.
3    |T (degC)       |-8.02              |Temperature in Celsius
4    |Tpot (K)       |265.4              |Temperature in Kelvin
5    |Tdew (degC)    |-8.9               |Temperature in Celsius relative to humidity. Dew Point is a measure of the absolute amount of water in the air, the DP is the temperature at which the air cannot hold all the moisture in it and water condenses.
6    |rh (%)         |93.3               |Relative Humidity is a measure of how saturated the air is with water vapor, the %RH determines the amount of water contained within collection objects.
7    |VPmax (mbar)   |3.33               |Saturation vapor pressure
8    |VPact (mbar)   |3.11               |Vapor pressure
9    |VPdef (mbar)   |0.22               |Vapor pressure deficit
10   |sh (g/kg)      |1.94               |Specific humidity
11   |H2OC (mmol/mol)|3.12               |Water vapor concentration
12   |rho (g/m ** 3) |1307.75            |Airtight
13   |wv (m/s)       |1.03               |Wind speed
14   |max. wv (m/s)  |1.75               |Maximum wind speed
15   |wd (deg)       |152.3              |Wind direction in degrees

In [2]:
"""
from zipfile import ZipFile

uri = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
zip_path = keras.utils.get_file(origin=uri, fname="jena_climate_2009_2016.csv.zip")
zip_file = ZipFile(zip_path)
zip_file.extractall()
"""

csv_path = "jena_climate_2009_2016.csv"
df = pd.read_csv(csv_path)

# Fix sensor error codes: -9999.0 in wind speed columns are equipment failures
# (18 rows in wv, 20 rows in max.wv — all from 2015-07-13 09:00–12:00).
# Replace with NaN so hourly .mean() resampling ignores them.
df[["wv (m/s)", "max. wv (m/s)"]] = df[["wv (m/s)", "max. wv (m/s)"]].replace(-9999.0, np.nan)

In [3]:
# Convert string to a proper DateTime so that
# matplotlib and the other components won't have to do it on every time
# Also, the date format in the CSV is European: DD.MM.YYYY HH:MM:SS (day comes first)
# Fix that by specifying the format explicitly:
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")

In [4]:
df.head(5)

# 3. Data Understanding and Exploration

## 3.1 Dataset Shape and Types

In [5]:
df.shape , df.dtypes
print('\nDataset Shape is:', df.shape)
print('\nDataset Types are:', df.dtypes)

## 3.2 Nan Values

In [6]:
df.isnull().sum() 

## 3.3 Check Duplicated Values

In [7]:
df.duplicated().sum()

## 3.4 Descriptive Stats 

In [8]:
df.describe(include=[np.number]).T

### 3.4.1 Variables Overview

In [9]:
titles = [
    "Pressure",
    "Temperature",
    "Temperature in Kelvin",
    "Temperature (dew point)",
    "Relative Humidity",
    "Saturation vapor pressure",
    "Vapor pressure",
    "Vapor pressure deficit",
    "Specific humidity",
    "Water vapor concentration",
    "Airtight",
    "Wind speed",
    "Maximum wind speed",
    "Wind direction in degrees",
]

feature_keys = [
    "p (mbar)",
    "T (degC)",
    "Tpot (K)",
    "Tdew (degC)",
    "rh (%)",
    "VPmax (mbar)",
    "VPact (mbar)",
    "VPdef (mbar)",
    "sh (g/kg)",
    "H2OC (mmol/mol)",
    "rho (g/m**3)",
    "wv (m/s)",
    "max. wv (m/s)",
    "wd (deg)",
]

colors = [
    "blue",
    "orange",
    "green",
    "red",
    "purple",
    "brown",
    "pink",
    "gray",
    "olive",
    "cyan",
]

date_time_key = "Date Time"


def show_raw_visualization(data):
    time_data = data[date_time_key]
    fig, axes = plt.subplots(
        nrows=7, ncols=2, figsize=(15, 20), dpi=80, facecolor="w", edgecolor="k"
    )
    for i in range(len(feature_keys)):
        key = feature_keys[i]
        c = colors[i % (len(colors))]
        t_data = data[key]
        t_data.index = time_data
        t_data.head()
        ax = t_data.plot(
            ax=axes[i // 2, i % 2],
            color=c,
            title="{} - {}".format(titles[i], key),
            rot=25,
        )
        ax.legend([titles[i]])
    plt.tight_layout()


show_raw_visualization(df)


## 3.5 Numeric Analysis

In [10]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", num_cols)
print("Total numeric:", len(num_cols))

### 3.5.1 Variables Histograms 

In [11]:
for c in num_cols:
    plt.figure()
    plt.hist(df[c].dropna(), bins=50)
    plt.title(f"Histogram - {c}")
    plt.xlabel(c)
    plt.ylabel("Count")
    plt.show()

### 3.5.2 Boxplots

In [12]:
for c in num_cols:
    plt.figure()
    plt.boxplot(df[c].dropna(), vert=False)
    plt.title(f"Boxplot - {c}")
    plt.xlabel(c)
    plt.show()

### 3.5.3 Outliers Detection

In [13]:
outlier_summary = []

for c in num_cols:
    s = df[c].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((s < lower) | (s > upper)).sum()
    outlier_summary.append([c, n_outliers, lower, upper])

outliers_df = pd.DataFrame(outlier_summary, columns=["col", "n_outliers", "iqr_low", "iqr_high"])
display(outliers_df.sort_values("n_outliers", ascending=False))

### 3.5.4 Correlation Between Variables

In [14]:
corr = df[num_cols].corr()

display(corr)

plt.figure(figsize=(8, 6))
plt.imshow(corr.values)
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.title("Correlation Matrix")
plt.colorbar()
plt.tight_layout()
plt.show()

### 3.5.6 Target (Correlation)

In [15]:
target = "T (degC)"
corr_target = corr[target].sort_values(ascending=False)

display(corr_target)

plt.figure()
plt.bar(corr_target.index, corr_target.values)
plt.xticks(rotation=90)
plt.title("Correlation with Temperature")
plt.ylabel("Correlation")
plt.tight_layout()
plt.show()

### 3.5.7 Temporal Evolution

In [16]:
plt.figure(figsize=(10, 4))
plt.plot(df["Date Time"], df["T (degC)"])
plt.title("Temperature over time")
plt.xlabel("Date")
plt.ylabel("T (degC)")
plt.show()

### 3.5.8 Mean by Day Hour

In [17]:
# Strip any accidental whitespace from column names (defensive, no-op if already clean)
df.columns = df.columns.str.strip()

In [18]:
df["hour"] = df["Date Time"].dt.hour
hourly_mean = df.groupby("hour")["T (degC)"].mean()

plt.figure()
plt.bar(hourly_mean.index, hourly_mean.values)
plt.title("Average Temperature by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Mean T (degC)")
plt.show()

### 3.5.9 Annual Sazonality

In [19]:
df["month"] = df["Date Time"].dt.month
monthly_mean = df.groupby("month")["T (degC)"].mean()

plt.figure()
plt.bar(monthly_mean.index, monthly_mean.values)
plt.title("Average Temperature by Month")
plt.xlabel("Month")
plt.ylabel("Mean T (degC)")
plt.show()

### 3.5.10 Target Relation with other Variables (Scatter)

In [20]:
target = "T (degC)"
exclude = {target, "hour", "month"}
x_cols = [c for c in num_cols if c not in exclude]

for x in x_cols:
    plt.figure()
    plt.scatter(df[x], df[target], s=1, alpha=0.3)
    plt.xlabel(x)
    plt.ylabel(target)
    plt.title(f"{target} vs {x}")
    plt.show()

### 3.5.11 Target Mean by 'bins' of Variables

In [21]:
target = "T (degC)"
exclude = {target, "hour", "month"}
x_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]

df_s = df.sample(50000, random_state=42)

for x in x_cols:
    # dividir x em 30 intervalos e ver média de T em cada intervalo
    bins = pd.cut(df_s[x], bins=30)
    m = df_s.groupby(bins)[target].mean()

    plt.figure()
    plt.plot(range(len(m)), m.values)
    plt.title(f"Mean {target} across {x} bins")
    plt.xlabel("Bin index (low -> high)")
    plt.ylabel(f"Mean {target}")
    plt.show()

### 3.5.11 Variables Mean by Hour

In [22]:
df["hour"] = df["Date Time"].dt.hour

vars_to_check = ["T (degC)", "rh (%)", "p (mbar)", "wv (m/s)"]
for v in vars_to_check:
    m = df.groupby("hour")[v].mean()
    plt.figure()
    plt.plot(m.index, m.values)
    plt.title(f"Mean {v} by hour")
    plt.xlabel("Hour")
    plt.ylabel(v)
    plt.show()

## 4. Data Understanding – Detailed Analysis

In [23]:
df = df.sort_values("Date Time").reset_index(drop=True)

target = "T (degC)"

### 4.1 Temporal Regularity and Gaps

In [24]:
# Check time deltas (frequency and gaps)
deltas = df["Date Time"].diff().dropna()
print("Most common deltas:")
display(deltas.value_counts().head(5))

mode_delta = deltas.value_counts().idxmax()
print("Mode delta:", mode_delta)

# Count gaps larger than the expected frequency
gaps = deltas[deltas > mode_delta]
print("Number of gaps larger than mode delta:", len(gaps))

# Show a few gaps (if any)
if len(gaps) > 0:
    idx = gaps.index[:10]
    display(df.loc[idx, ["Date Time"]].assign(delta=gaps.loc[idx].values))

## 4.2 Seasonal Patterns
### 4.2.1 Stationarity (Basic Check)

In [25]:
# Rolling mean and std for temperature (7-day window)
w = 7 * 24 * 6  # 7 days with 10-min data
t = df.set_index("Date Time")["T (degC)"]

roll_mean = t.rolling(w).mean()
roll_std = t.rolling(w).std()

plt.figure(figsize=(10, 4))
plt.plot(t, alpha=0.4, label="Temperature")
plt.plot(roll_mean, label="Rolling Mean (7 days)")
plt.plot(roll_std, label="Rolling Std (7 days)")
plt.title("Temperature - Rolling Mean and Std (7 days)")
plt.legend()
plt.show()

### 4.2.2 Preview of Hourly Resampling

In [26]:
feature_keys = [
    "p (mbar)", "T (degC)", "Tpot (K)", "Tdew (degC)", "rh (%)",
    "VPmax (mbar)", "VPact (mbar)", "VPdef (mbar)", "sh (g/kg)",
    "H2OC (mmol/mol)", "rho (g/m**3)", "wv (m/s)", "max. wv (m/s)", "wd (deg)"
]

df_hourly = (
    df.set_index("Date Time")[feature_keys]
      .resample("1h")   # <- lowercase h
      .mean()
      .reset_index()
)

print("Hourly shape:", df_hourly.shape)
display(df_hourly.head())

In [27]:
print("Hourly missing timestamps:", df_hourly["Date Time"].isna().sum())
print("NaNs per column (hourly):")
display(df_hourly.isna().sum())

In [28]:
end = df["Date Time"].max()
start = end - pd.Timedelta(days=7)

raw_7d = df[(df["Date Time"] >= start) & (df["Date Time"] <= end)]
hourly_7d = df_hourly[(df_hourly["Date Time"] >= start) & (df_hourly["Date Time"] <= end)]

plt.figure(figsize=(10, 4))
plt.plot(raw_7d["Date Time"], raw_7d["T (degC)"], alpha=0.4, label="Raw (10 min)")
plt.plot(hourly_7d["Date Time"], hourly_7d["T (degC)"], label="Hourly (mean)")
plt.title("Temperature: Raw vs Hourly (7 days)")
plt.xlabel("Date")
plt.ylabel("T (degC)")
plt.legend()
plt.show()

### 4.3 Find rows (hours) with NaNs

In [29]:
na_rows = df_hourly[df_hourly.isna().any(axis=1)].copy()
print("Hourly rows with NaN:", len(na_rows))
display(na_rows.head(10))
display(na_rows.tail(10))

#### 4.3.1 Remove Nan 

In [30]:
# Clean hourly data by dropping hours with no observations
df_hourly_clean = df_hourly.dropna().reset_index(drop=True)

print("Hourly before cleaning:", df_hourly.shape)
print("Hourly after cleaning: ", df_hourly_clean.shape)

# Sanity check
print("Remaining NaNs:", df_hourly_clean.isna().sum().sum())

## 5. Data Preparation

### 5.1 Dataset Scope and Columns

#### 5.1.1 Define target, inputs and frequency

In [31]:
# Target variable
target_col = "T (degC)"

# Input features (Jena standard set)
feature_keys = ["T (degC)", "p (mbar)", "rh (%)", "wv (m/s)", "max. wv (m/s)", "wd (deg)"]

# Final working dataframe (hourly, clean)
df_work = df_hourly_clean[["Date Time"] + feature_keys].copy()

print("df_work shape:", df_work.shape)
display(df_work.head())

#### 5.1.2 Sanity checks (no NaNs, correct dtypes)

In [32]:
print("NaNs per column:")
display(df_work.isna().sum())

print("\nDtypes:")
display(df_work.dtypes)

#### 5.1.3 Confirm frequency is hourly

In [33]:
# Check time delta between consecutive rows
deltas = df_work["Date Time"].diff().dropna()
print("Most common deltas:")
display(deltas.value_counts().head(3))

#### 5.1.4 Set Date Time as index

In [34]:
df_work = df_work.set_index("Date Time")
print("Index type:", type(df_work.index))
display(df_work.head())

### 5.2 Temporal Split (Train / Validation / Test)

#### 5.2.1 Split Proportions

In [35]:
# Split ratios
train_ratio = 0.70
val_ratio   = 0.15
test_ratio  = 0.15

n_total = len(df_work)
n_train = int(n_total * train_ratio)
n_val   = int(n_total * val_ratio)

print("Total samples:", n_total)
print("Train size:", n_train)
print("Val size:", n_val)
print("Test size:", n_total - n_train - n_val)

In [36]:
#### 4.2.2 Criar os conjuntos (contíguos no tempo)

In [37]:
df_train = df_work.iloc[:n_train]
df_val   = df_work.iloc[n_train:n_train + n_val]
df_test  = df_work.iloc[n_train + n_val:]

print("Train period:", df_train.index.min(), "->", df_train.index.max())
print("Val period:  ", df_val.index.min(),   "->", df_val.index.max())
print("Test period: ", df_test.index.min(),  "->", df_test.index.max())

print("\nShapes:")
print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)

#### 4.2.3 Visual target over splits

In [38]:
plt.figure(figsize=(10, 4))
plt.plot(df_train.index, df_train[target_col], label="Train")
plt.plot(df_val.index,   df_val[target_col],   label="Val")
plt.plot(df_test.index,  df_test[target_col],  label="Test")
plt.title("Temporal Split of Temperature")
plt.xlabel("Date")
plt.ylabel("T (degC)")
plt.legend()
plt.show()

### 5.3 Feature Engineering

In [39]:
def add_time_features(df_in):
    df = df_in.copy()
    # time parts
    hour = df.index.hour
    doy  = df.index.dayofyear

    # cyclical encoding
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)

    df["doy_sin"]  = np.sin(2 * np.pi * doy / 365.25)
    df["doy_cos"]  = np.cos(2 * np.pi * doy / 365.25)

    # wind direction is circular
    wd = df["wd (deg)"].astype(float)
    df["wd_sin"] = np.sin(2 * np.pi * wd / 360.0)
    df["wd_cos"] = np.cos(2 * np.pi * wd / 360.0)

    return df

df_train_fe = add_time_features(df_train)
df_val_fe   = add_time_features(df_val)
df_test_fe  = add_time_features(df_test)

print(df_train_fe.shape, df_val_fe.shape, df_test_fe.shape)

## 5.4 Scaling

In [40]:
from sklearn.preprocessing import StandardScaler

# Choose features to scale (all columns)
feat_cols = df_train_fe.columns.tolist()

scaler = StandardScaler()
scaler.fit(df_train_fe[feat_cols])

train_scaled = scaler.transform(df_train_fe[feat_cols])
val_scaled   = scaler.transform(df_val_fe[feat_cols])
test_scaled  = scaler.transform(df_test_fe[feat_cols])

# Back to DataFrame (keep index)
df_train_s = pd.DataFrame(train_scaled, index=df_train_fe.index, columns=feat_cols)
df_val_s   = pd.DataFrame(val_scaled,   index=df_val_fe.index,   columns=feat_cols)
df_test_s  = pd.DataFrame(test_scaled,  index=df_test_fe.index,  columns=feat_cols)

print("NaNs after scaling:", df_train_s.isna().sum().sum(), df_val_s.isna().sum().sum(), df_test_s.isna().sum().sum())

## 5.5 Windowing (L=120, H=24)

In [41]:
from numpy.lib.stride_tricks import sliding_window_view

L = 120  # input window (hours)
H = 24   # forecast horizon (hours)

def make_windows(df_scaled, target_name, L, H):
    # Cast to float32 once here — avoids repeated float64→float32
    # conversion inside TensorFlow on every batch during training.
    data = df_scaled.values.astype(np.float32)
    target_idx = df_scaled.columns.get_loc(target_name)
    n_samples = len(data) - L - H

    # Build X windows fully vectorized using stride tricks — no Python loop.
    # sliding_window_view(data, L, axis=0) → (n-L+1, n_features, L)
    # transpose(0,2,1)                     → (n-L+1, L, n_features)
    # [:n_samples]                          → (n_samples, L, n_features)
    X = sliding_window_view(data, window_shape=L, axis=0).transpose(0, 2, 1)[:n_samples].copy()

    # Build y windows: target column only, H steps ahead of each X window.
    # sliding_window_view on target[L:] → (n-L-H+1, H), take [:n_samples]
    y = sliding_window_view(data[L:, target_idx], window_shape=H)[:n_samples].copy()

    return X, y

X_train, y_train = make_windows(df_train_s, target_col, L, H)
X_val,   y_val   = make_windows(df_val_s,   target_col, L, H)
X_test,  y_test  = make_windows(df_test_s,  target_col, L, H)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)
print("dtype:", X_train.dtype)

#### 5.5.1 Baseline: Persistence

In [42]:
# persistence: predict next 24 hours as last observed temperature in the input window
target_idx = df_test_s.columns.get_loc(target_col)

last_T = X_test[:, -1, target_idx]  # last temp in each window
y_pred_persist = np.repeat(last_T[:, None], y_test.shape[1], axis=1)

print(y_pred_persist.shape)

#### 5.5.2 Metrics

In [43]:
def eval_multihorizon(y_true, y_pred):
    mae = mean_absolute_error(y_true.reshape(-1), y_pred.reshape(-1))
    rmse = np.sqrt(mean_squared_error(y_true.reshape(-1), y_pred.reshape(-1)))
    return mae, rmse

mae_p, rmse_p = eval_multihorizon(y_test, y_pred_persist)
print("Persistence - MAE:", mae_p, "RMSE:", rmse_p)

#### 5.5.3 MAE by horizont

In [44]:
mae_by_h = np.mean(np.abs(y_test - y_pred_persist), axis=0)

plt.figure()
plt.plot(range(1, len(mae_by_h) + 1), mae_by_h)
plt.title("Persistence baseline - MAE by horizon")
plt.xlabel("Horizon (hours ahead)")
plt.ylabel("MAE (scaled)")
plt.show()

## 6. Model: GRU

In [45]:
print("TF version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs found:", gpus)

if gpus:
    # Avoid TF grabbing all GPU memory at once
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU is available and memory growth is enabled.")
else:
    print("No GPU found. Training will run on CPU.")

### 6.1 Setup: reproducibility

In [46]:
tf.random.set_seed(42)
np.random.seed(42)

H = y_train.shape[1]
L = X_train.shape[1]
n_features = X_train.shape[2]

### 6.2 Build function Hyperparameters

In [47]:
def build_gru_model(
    L, n_features, H,
    units1=64,
    units2=32,
    n_layers=1,
    dropout=0.2,
    l2=0.0,
    dense_units=0,
    dense_activation="relu",
    learning_rate=1e-3,
    clipnorm=1.0,
    optimizer_name="adam",
    # --- new (optional) params, defaults keep old behavior ---
    units3=32,
    weight_decay=1e-5,          # used only if adamw
    loss_name="mse",            # mse/mae/huber1/huber2
    gaussian_noise_std=0.0,     # 0 disables
):
    reg = keras.regularizers.l2(l2) if l2 and l2 > 0 else None

    inputs = keras.Input(shape=(L, n_features))
    x = inputs

    # Optional input noise
    if gaussian_noise_std and gaussian_noise_std > 0:
        x = layers.GaussianNoise(gaussian_noise_std)(x)

    # GRU stack (supports 1/2/3 layers).
    # recurrent_dropout=0.0 is required to use the fast cuDNN GRU kernel on GPU.
    x = layers.GRU(
        units1,
        return_sequences=(n_layers >= 2),
        dropout=dropout,
        recurrent_dropout=0.0,
        kernel_regularizer=reg
    )(x)

    if n_layers >= 2:
        x = layers.GRU(
            units2,
            return_sequences=(n_layers == 3),
            dropout=dropout,
            recurrent_dropout=0.0,
            kernel_regularizer=reg
        )(x)

    if n_layers == 3:
        x = layers.GRU(
            units3,
            return_sequences=False,
            dropout=dropout,
            recurrent_dropout=0.0,
            kernel_regularizer=reg
        )(x)

    # Optional dense head with extra activations
    if dense_units and dense_units > 0:
        if dense_activation == "leaky_relu":
            x = layers.Dense(dense_units, kernel_regularizer=reg)(x)
            x = layers.LeakyReLU(negative_slope=0.1)(x)
        else:
            x = layers.Dense(dense_units, activation=dense_activation, kernel_regularizer=reg)(x)
        x = layers.Dropout(dropout)(x)

    outputs = layers.Dense(H)(x)  # linear output
    model = keras.Model(inputs, outputs)

    # Optimizer
    if optimizer_name.lower() == "adamw":
        opt = keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            clipnorm=clipnorm
        )
    else:
        opt = keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=clipnorm)

    # Loss
    if loss_name == "mae":
        loss = "mae"
    elif loss_name == "huber1":
        loss = keras.losses.Huber(delta=1.0)
    elif loss_name == "huber2":
        loss = keras.losses.Huber(delta=2.0)
    else:
        loss = "mse"

    model.compile(optimizer=opt, loss=loss, metrics=["mae"])
    return model

### 6.3 Training wrapper + callbacks

In [48]:
def train_one_config(cfg, X_train, y_train, X_val, y_val, max_epochs=60, fit_verbose=0):
    model = build_gru_model(
        L=L, n_features=n_features, H=H,
        units1=cfg["units1"],
        units2=cfg["units2"],
        n_layers=cfg["n_layers"],
        dropout=cfg["dropout"],
        l2=cfg["l2"],
        dense_units=cfg["dense_units"],
        dense_activation=cfg["dense_activation"],
        learning_rate=cfg["lr"],
        clipnorm=cfg["clipnorm"],
        optimizer_name=cfg["optimizer"],
        # new params
        units3=cfg.get("units3", 32),
        weight_decay=cfg.get("weight_decay", 0.0),
        loss_name=cfg.get("loss_name", "mse"),
        gaussian_noise_std=cfg.get("gaussian_noise_std", 0.0),
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=max_epochs,
        batch_size=cfg["batch_size"],
        verbose=fit_verbose,
        callbacks=callbacks
    )

    best_val_loss = float(np.min(history.history["val_loss"]))
    best_val_mae  = float(np.min(history.history["val_mae"]))
    return model, history, best_val_loss, best_val_mae

### 6.4 Random hyperparameter search loop

In [49]:
# Custom pruning callback — avoids dependency on optuna.integration.*
# After each epoch it reports val_loss to Optuna; if Optuna's Hyperband pruner
# decides this trial is unlikely to beat the current best, it raises TrialPruned
# and Keras stops training immediately, saving compute.
class OptunaPruningCallback(keras.callbacks.Callback):
    def __init__(self, trial):
        super().__init__()
        self.trial = trial

    def on_epoch_end(self, epoch, logs=None):
        val_loss = logs.get("val_loss")
        if val_loss is None:
            return
        self.trial.report(val_loss, epoch)
        if self.trial.should_prune():
            raise optuna.exceptions.TrialPruned()


# Mutable dict to keep the best Keras model across trials.
# Optuna tracks hyperparameters and val_loss, but not model objects.
best_holder = {"val_loss": np.inf, "model": None, "cfg": None, "history": None}


def objective(trial):
    # ── Sample hyperparameters via TPE (Bayesian) ────────────────────────────
    n_layers    = trial.suggest_int("n_layers", 1, 3)
    units1      = trial.suggest_categorical("units1",      [64, 96, 128, 192])
    units2      = trial.suggest_categorical("units2",      [32, 64, 96, 128])
    units3      = trial.suggest_categorical("units3",      [32, 64, 96])
    dropout     = trial.suggest_categorical("dropout",     [0.0, 0.1, 0.2, 0.3, 0.4])
    l2          = trial.suggest_categorical("l2",          [0.0, 1e-6, 1e-5, 1e-4])
    dense_units = trial.suggest_categorical("dense_units", [0, 64, 128, 256])
    dense_act   = trial.suggest_categorical("dense_activation", ["relu", "gelu", "elu", "leaky_relu"])
    lr          = trial.suggest_categorical("lr",          [2e-3, 1e-3, 5e-4, 3e-4, 2e-4, 1e-4])
    batch_size  = trial.suggest_categorical("batch_size",  [128, 256, 512])
    clipnorm    = trial.suggest_categorical("clipnorm",    [0.5, 1.0, 2.0, 5.0])
    opt_name    = trial.suggest_categorical("optimizer",   ["adam", "adamw"])
    weight_decay= trial.suggest_categorical("weight_decay",[0.0, 1e-6, 1e-5, 1e-4])
    loss_name   = trial.suggest_categorical("loss_name",   ["mse", "mae", "huber1", "huber2"])
    noise_std   = trial.suggest_categorical("gaussian_noise_std", [0.0, 0.01, 0.05])

    # ── Apply same constraints as the random search ──────────────────────────
    if units2 > units1:   units2 = units1
    if units3 > units2:   units3 = units2
    if opt_name == "adam": weight_decay = 0.0
    if n_layers == 1 and dense_units > 0 and random.random() < 0.7:
        dense_units = 0

    cfg = dict(
        n_layers=n_layers, units1=units1, units2=units2, units3=units3,
        dropout=dropout, l2=l2, dense_units=dense_units, dense_activation=dense_act,
        lr=lr, batch_size=batch_size, clipnorm=clipnorm, optimizer=opt_name,
        weight_decay=weight_decay, loss_name=loss_name, gaussian_noise_std=noise_std,
    )

    model = build_gru_model(
        L=L, n_features=n_features, H=H,
        units1=units1, units2=units2, units3=units3,
        n_layers=n_layers, dropout=dropout, l2=l2,
        dense_units=dense_units, dense_activation=dense_act,
        learning_rate=lr, clipnorm=clipnorm,
        optimizer_name=opt_name, weight_decay=weight_decay,
        loss_name=loss_name, gaussian_noise_std=noise_std,
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
        OptunaPruningCallback(trial),
    ]

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=40,
        batch_size=batch_size,
        verbose=0,
        callbacks=callbacks,
    )

    val_loss = float(np.min(history.history["val_loss"]))
    val_mae  = float(np.min(history.history["val_mae"]))

    trial.set_user_attr("val_mae", val_mae)

    if val_loss < best_holder["val_loss"]:
        best_holder.update({"val_loss": val_loss, "model": model, "history": history, "cfg": cfg})

    return val_loss


### 6.5 Run Random Search

In [ ]:
from IPython.display import clear_output

def progress_callback(study, trial):
    """Live chart redrawn after every trial (complete or pruned)."""
    clear_output(wait=True)
    completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pruned    = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    n_done    = len(study.trials)

    state_str = ("PRUNED" if trial.state == optuna.trial.TrialState.PRUNED
                 else f"val_loss={trial.value:.4f}")
    print(f"[{n_done}/60] Trial #{trial.number} → {state_str}")
    if completed:
        print(f"  Complete: {len(completed)} | Pruned: {len(pruned)} | Best so far: {study.best_value:.4f}")

    if len(completed) < 2:
        return

    losses       = [t.value for t in completed]
    running_best = pd.Series(losses).cummin()
    durations    = [t.duration.total_seconds() for t in completed if t.duration]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    ax.scatter(range(1, len(losses) + 1), losses, alpha=0.5, s=30, label="Trial val_loss")
    ax.plot(range(1, len(losses) + 1), running_best, color="red", linewidth=2, label="Running best")
    ax.set_title(f"Optuna Progress [{len(completed)} complete, {len(pruned)} pruned]"
                 f" | Best: {study.best_value:.4f}")
    ax.set_xlabel("Completed trial index")
    ax.set_ylabel("val_loss (MSE, scaled)")
    ax.legend()

    ax = axes[1]
    ax.bar(range(1, len(durations) + 1), durations, color="steelblue", alpha=0.7)
    if durations:
        ax.axhline(np.mean(durations), color="red", linestyle="--",
                   label=f"Mean: {np.mean(durations):.1f}s")
    ax.set_title("Trial Duration")
    ax.set_xlabel("Completed trial index")
    ax.set_ylabel("Duration (s)")
    ax.legend()

    plt.tight_layout()
    plt.show()


# TPE sampler: Bayesian optimisation via Tree-structured Parzen Estimators.
# After an initial random phase (~10 trials) it models the loss landscape
# and focuses sampling on promising regions — unlike pure random search.
#
# HyperbandPruner: kills unpromising trials early based on epoch-level val_loss.
# min_resource=3  → every trial runs at least 3 epochs before pruning kicks in.
# max_resource=40 → matches our max_epochs.
# reduction_factor=3 → each bracket keeps the top 1/3 of trials.
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.HyperbandPruner(min_resource=3, max_resource=40, reduction_factor=3),
)

study.optimize(objective, n_trials=60, callbacks=[progress_callback])

print("\nOptimization complete!")
print(f"Best trial #{study.best_trial.number} — val_loss: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")


In [ ]:
# Build results_df from completed Optuna trials in the same column format
# as the random search version so all downstream evaluation cells work unchanged.
rows = []
for t in study.trials:
    if t.state != optuna.trial.TrialState.COMPLETE:
        continue
    row = {
        "name":           f"trial_{t.number:02d}",
        "best_val_loss":  t.value,
        "best_val_mae":   t.user_attrs.get("val_mae", np.nan),
        "train_time_sec": round(t.duration.total_seconds(), 1) if t.duration else np.nan,
        **t.params,
    }
    rows.append(row)

n_pruned = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
results_df = pd.DataFrame(rows).sort_values("best_val_loss").reset_index(drop=True)

print(f"Completed trials: {len(rows)} | Pruned: {n_pruned}")
display(results_df.head(10))

# Compatibility aliases — downstream cells reference these names.
# Inject "name" into cfg so title labels in evaluation cells work unchanged.
best_cfg_named = {**best_holder["cfg"], "name": f"trial_{study.best_trial.number:02d}"}
best = {
    "val_loss": best_holder["val_loss"],
    "model":    best_holder["model"],
    "history":  best_holder["history"],
    "cfg":      best_cfg_named,
}
best_gru = best_holder["model"]

### 6.6 Plot training curves (best GRU)

In [ ]:
hist = best["history"].history

plt.figure()
plt.plot(hist["loss"], label="train_loss")
plt.plot(hist["val_loss"], label="val_loss")
plt.title(f"Best GRU - Loss ({best['cfg']['name']})")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()

plt.figure()
plt.plot(hist["mae"], label="train_mae")
plt.plot(hist["val_mae"], label="val_mae")
plt.title(f"Best GRU - MAE ({best['cfg']['name']})")
plt.xlabel("Epoch")
plt.ylabel("MAE (scaled)")
plt.legend()
plt.show()

### 6.7 Evaluate best GRU on TEST (scaled)

In [ ]:
best_gru = best["model"]
y_pred_gru_scaled = best_gru.predict(X_test, verbose=0)

yt = y_test.reshape(-1)
yp = y_pred_gru_scaled.reshape(-1)

mae = mean_absolute_error(yt, yp)
rmse = np.sqrt(mean_squared_error(yt, yp))
r2 = r2_score(yt, yp)

# MAPE on scaled units is not meaningful; we'll compute MAPE in °C after inverse-scaling.
print("TEST (scaled) - MAE:", mae, "RMSE:", rmse, "R2:", r2)

### 6.8 Convert predictions to °C + compute MAE/RMSE/MAPE/R²

In [ ]:
def inverse_scale_target(y_scaled, scaler, feature_names, target_col):
    t_idx = feature_names.index(target_col)
    mean = scaler.mean_[t_idx]
    std  = scaler.scale_[t_idx]
    return y_scaled * std + mean

y_test_c = inverse_scale_target(y_test, scaler, feat_cols, target_col)
y_pred_c = inverse_scale_target(y_pred_gru_scaled, scaler, feat_cols, target_col)

#### 6.8.1 Metrics in °C (global + MAPE)

In [ ]:
eps = 1e-6
yt = y_test_c.reshape(-1)
yp = y_pred_c.reshape(-1)

mae_c = mean_absolute_error(yt, yp)
rmse_c = np.sqrt(mean_squared_error(yt, yp))
r2_c = r2_score(yt, yp)

print("TEST (°C) - MAE:", mae_c, "RMSE:", rmse_c, "R2:", r2_c)

### 6.9 MAE by horizon (°C)

In [ ]:
H = y_test.shape[1]
mae_by_h = np.mean(np.abs(y_test_c - y_pred_c), axis=0)

plt.figure()
plt.plot(range(1, H+1), mae_by_h)
plt.title("Best GRU - MAE by Horizon (°C)")
plt.xlabel("Horizon (hours ahead)")
plt.ylabel("MAE (°C)")
plt.show()

In [ ]:
results_df.to_csv("gru_random_search_results.csv", index=False)

### 7. Overview table + top/bottom configs

In [ ]:
display(results_df.head(10))     # top 10
display(results_df.tail(10))     # worst 10
print("Configs tested:", len(results_df))
print("Best val_loss:", results_df["best_val_loss"].min())
print("Best val_mae:", results_df["best_val_mae"].min())

#### 7.1 Histogram: distribution of validation performance

In [ ]:
plt.figure()
plt.hist(results_df["best_val_loss"], bins=20)
plt.title("Distribution of best validation loss (MSE)")
plt.xlabel("best_val_loss")
plt.ylabel("count")
plt.show()

plt.figure()
plt.hist(results_df["best_val_mae"], bins=20)
plt.title("Distribution of best validation MAE")
plt.xlabel("best_val_mae")
plt.ylabel("count")
plt.show()

In [ ]:
# Search progress chart
# results_df is sorted by val_loss; we recover search order via the config names (gru_rs_01...gru_rs_60)
results_order = results_df.sort_values("name").reset_index(drop=True)
x = range(1, len(results_order) + 1)
running_best = results_order["best_val_loss"].cummin()
best_idx = int(results_order["best_val_loss"].idxmin()) + 1  # 1-indexed position

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: val_loss per config + running best
ax = axes[0]
ax.scatter(x, results_order["best_val_loss"], alpha=0.5, s=30, label="Config val_loss")
ax.plot(x, running_best, color="red", linewidth=2, label="Running best")
ax.axvline(best_idx, color="red", linestyle="--", alpha=0.4, label=f"Best found at #{best_idx}")
ax.set_title("Random Search Progress")
ax.set_xlabel("Config index (search order)")
ax.set_ylabel("Best val_loss")
ax.legend()

# Right: val_loss by learning rate
ax = axes[1]
results_df.boxplot(column="best_val_loss", by="lr", ax=ax)
ax.set_title("Val loss by learning rate")
plt.suptitle("")
ax.set_xlabel("Learning rate")
ax.set_ylabel("best_val_loss")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

#### 7.2 Scatter: performance vs training time (Pareto intuition)

In [ ]:
plt.figure()
plt.scatter(results_df["train_time_sec"], results_df["best_val_loss"], alpha=0.7)
plt.title("Validation loss vs training time")
plt.xlabel("train_time_sec")
plt.ylabel("best_val_loss (lower is better)")
plt.show()

#### 7.4 Important Hyperparameters

In [ ]:
plt.figure()
results_df.boxplot(column="best_val_loss", by="n_layers")
plt.title("Val loss by number of GRU layers")
plt.suptitle("")
plt.xlabel("n_layers")
plt.ylabel("best_val_loss")
plt.show()

##### 7.4.1 By Dropout

In [ ]:
plt.figure()
results_df.boxplot(column="best_val_loss", by="dropout")
plt.title("Val loss by dropout")
plt.suptitle("")
plt.xlabel("dropout")
plt.ylabel("best_val_loss")
plt.show()

##### 7.4.1 By Optimizer

In [ ]:
plt.figure()
results_df.boxplot(column="best_val_loss", by="optimizer")
plt.title("Val loss by optimizer")
plt.suptitle("")
plt.xlabel("optimizer")
plt.ylabel("best_val_loss")
plt.show()

## 8. Horizon-wise comparison: baseline vs best GRU (°C)

In [ ]:
# Convert persistence baseline to °C
y_pred_persist_c = inverse_scale_target(y_pred_persist, scaler, feat_cols, target_col)

H = y_test.shape[1]
mae_h_p = np.mean(np.abs(y_test_c - y_pred_persist_c), axis=0)
mae_h_g = np.mean(np.abs(y_test_c - y_pred_c), axis=0)

plt.figure()
plt.plot(range(1, H+1), mae_h_p, label="Persistence")
plt.plot(range(1, H+1), mae_h_g, label="Best GRU")
plt.title("MAE by horizon (°C) - Baseline vs GRU")
plt.xlabel("Horizon (hours ahead)")
plt.ylabel("MAE (°C)")
plt.legend()
plt.show()

### 8.1 Error distribution (residuals) on test (°C)

In [ ]:
res = (y_test_c - y_pred_c).reshape(-1)

plt.figure()
plt.hist(res, bins=60)
plt.title("Residual distribution (True - Pred) in °C")
plt.xlabel("Residual (°C)")
plt.ylabel("Count")
plt.show()

# Absolute error distribution
abs_err = np.abs(res)
plt.figure()
plt.hist(abs_err, bins=60)
plt.title("Absolute error distribution (°C)")
plt.xlabel("|Error| (°C)")
plt.ylabel("Count")
plt.show()

### 8.2 Plot actual vs predicted for a continuous slice (h=1)

In [ ]:
start = 0
n = 7 * 24  # 1 week of windows

y_true_1 = y_test_c[start:start+n, 0]
y_pred_1 = y_pred_c[start:start+n, 0]

plt.figure(figsize=(10,4))
plt.plot(y_true_1, label="True (h=1)")
plt.plot(y_pred_1, label="Pred (h=1)")
plt.title("1-hour ahead forecast (1-week slice, °C)")
plt.xlabel("Test window index")
plt.ylabel("T (degC)")
plt.legend()
plt.show()

### 8.3 Plot 24-hour trajectories for a few examples

In [ ]:
examples = [50, 200, 500]  # pick any indices within test

for i in examples:
    plt.figure(figsize=(10,4))
    plt.plot(range(1, H+1), y_test_c[i], label="True")
    plt.plot(range(1, H+1), y_pred_c[i], label="Pred")
    plt.title(f"24-hour forecast trajectory example (test idx={i})")
    plt.xlabel("Horizon (hours ahead)")
    plt.ylabel("T (degC)")
    plt.legend()
    plt.show()

### 8.4 Learning curves for top-3 configs

In [ ]:
# Retrain top-3 configs to visualise their learning curves.
# config_keys comes from the Optuna study so all sampled parameters are included.
config_keys = list(study.best_params.keys())

top3 = results_df.head(3).to_dict(orient="records")
histories = []

for cfg_row in top3:
    cfg = {k: cfg_row[k] for k in config_keys if k in cfg_row}
    model, history, best_val_loss, best_val_mae = train_one_config(
        cfg, X_train, y_train, X_val, y_val, max_epochs=40
    )
    histories.append((cfg_row["name"], history.history))

for name, h in histories:
    plt.figure()
    plt.plot(h["val_loss"],  label="val_loss")
    plt.plot(h["loss"],      label="train_loss")
    plt.title(f"Loss curves - {name}")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()
    plt.show()


## 9. Forecasting

In [ ]:
# 2.1) Build last input window (L hours) from the end of the dataset
df_full = df_work.copy()  # df_work: hourly clean, index=Date Time, columns=features (unscaled)

# If you engineered features during training, apply the same here:
df_full_fe = add_time_features(df_full)

# Keep the same columns and order used in scaling/training
df_full_fe = df_full_fe[feat_cols].copy()

# Take last L rows
last_window = df_full_fe.iloc[-L:].copy()

# Scale
last_window_scaled = scaler.transform(last_window.values)

# Model input shape: (1, L, n_features)
X_last = last_window_scaled.reshape(1, L, len(feat_cols))

# Predict next 24h (scaled)
y_next24_scaled = best_gru.predict(X_last, verbose=0)  # shape (1, 24)

# Inverse scale to °C
y_next24_c = inverse_scale_target(y_next24_scaled, scaler, feat_cols, target_col).flatten()

# Build future datetime index
last_time = df_full_fe.index[-1]
future_index_24 = pd.date_range(start=last_time + pd.Timedelta(hours=1), periods=H, freq="1h")

# Plot forecast
plt.figure(figsize=(10,4))
plt.plot(future_index_24, y_next24_c, marker="o")
plt.title("Forecast - Next 24 hours (Temperature)")
plt.xlabel("Date Time")
plt.ylabel("T (degC)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# Show as table
forecast_24h = pd.DataFrame({"Date Time": future_index_24, "T_forecast_degC": y_next24_c})
display(forecast_24h.head(10))
display(forecast_24h.tail(10))

### 9.2 5 days Forecasting

In [ ]:
def forecast_rolling_5days(df_full_raw, best_model, scaler, feat_cols, target_col, L=120, H=24, days=5, lag_hours=168):
    steps = days * 24  # total future hours
    n_blocks = steps // H

    df_sim = df_full_raw.copy()
    preds_all = []
    last_time = df_sim.index[-1]

    for b in range(n_blocks):
        # Recompute time features over current (growing) df_sim
        df_sim_fe = add_time_features(df_sim)

        # Use last L hours as model input
        X_win = df_sim_fe[feat_cols].iloc[-L:].values
        X_win_s = scaler.transform(X_win).reshape(1, L, len(feat_cols))

        y_block_s = best_model.predict(X_win_s, verbose=0)  # (1, H)
        y_block_c = inverse_scale_target(y_block_s, scaler, feat_cols, target_col).flatten()

        # Future timestamps for this block
        start = last_time + pd.Timedelta(hours=1)
        idx = pd.date_range(start=start, periods=H, freq="1h")

        preds_all.append(pd.DataFrame({"Date Time": idx, "T_forecast_degC": y_block_c}))

        # Seasonal-naive imputation for exogenous features:
        # For each future hour t, use the observed values from lag_hours ago (default: 168h = 7 days).
        # This is physically motivated: atmospheric variables (pressure, humidity, wind)
        # exhibit strong weekly periodicity, making same-hour-last-week a better
        # proxy than freezing the last observed value indefinitely.
        for t, yv in zip(idx, y_block_c):
            lag_time = t - pd.Timedelta(hours=lag_hours)
            if lag_time in df_sim.index:
                new_row = df_sim.loc[[lag_time]].copy()
            else:
                new_row = df_sim.iloc[-1:].copy()  # fallback if lag not in history
            new_row.index = [t]
            new_row[target_col] = yv  # temperature always from model prediction
            df_sim = pd.concat([df_sim, new_row])

        last_time = idx[-1]

    return pd.concat(preds_all, ignore_index=True)

forecast_5d = forecast_rolling_5days(df_work, best_gru, scaler, feat_cols, target_col, L=L, H=H, days=5)
display(forecast_5d.head())
display(forecast_5d.tail())

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(forecast_5d["Date Time"], forecast_5d["T_forecast_degC"])
plt.title("Forecast - Next 5 days (Temperature) [rolling]")
plt.xlabel("Date Time")
plt.ylabel("T (degC)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()